In [5]:
import pandas as pd

wiki_df = pd.read_json("wiki_csai.jsonl", lines=True)
wiki_df["source"] = "wiki_csai"

finance_df = pd.read_json("finance.jsonl", lines=True)
finance_df["source"] = "finance"

hc3 = pd.concat([wiki_df, finance_df], ignore_index=True)

print(hc3.shape)
print(hc3.head())
print(hc3["source"].value_counts())

(4775, 4)
                                            question  \
0          Please explain what is "Animal cognition"   
1        Please explain what is "Human intelligence"   
2  Please explain what is "Oxford English Diction...   
3   Please explain what is "Oxford University Press"   
4           Please explain what is "AI applications"   

                                       human_answers  \
0  [Animal cognition encompasses the mental capac...   
1  [Human intelligence is the intellectual capabi...   
2  [The Oxford English Dictionary (OED) is the fi...   
3  [Oxford University Press (OUP) is the universi...   
4  [Artificial intelligence (AI) has been used in...   

                                     chatgpt_answers     source  
0  [Animal cognition refers to the mental capacit...  wiki_csai  
1  [Human intelligence is the mental ability to t...  wiki_csai  
2  [The Oxford English Dictionary (OED) is a comp...  wiki_csai  
3  [Oxford University Press (OUP) is a department...

In [10]:
import pandas as pd

wiki_df = pd.read_json("wiki_csai.jsonl", lines=True)
wiki_df["source"] = "wiki_csai"

finance_df = pd.read_json("finance.jsonl", lines=True)
finance_df["source"] = "finance"

hc3 = pd.concat([wiki_df, finance_df], ignore_index=True)
hc3["id"] = hc3.index.astype(str)   # <-- ADD THIS LINE

print(hc3.shape)
print(hc3.columns.tolist())
print(hc3["source"].value_counts())

(4775, 5)
['question', 'human_answers', 'chatgpt_answers', 'source', 'id']
source
finance      3933
wiki_csai     842
Name: count, dtype: int64


In [11]:
# explode human answers
human_long = hc3[["id", "question", "source", "human_answers"]].explode("human_answers")
human_long = human_long.rename(columns={"human_answers": "text"})
human_long["label"] = "human"

# explode ChatGPT answers
ai_long = hc3[["id", "question", "source", "chatgpt_answers"]].explode("chatgpt_answers")
ai_long = ai_long.rename(columns={"chatgpt_answers": "text"})
ai_long["label"] = "ai"

# combine
hc3_long = pd.concat([human_long, ai_long], ignore_index=True)
hc3_long = hc3_long.dropna(subset=["text"]).reset_index(drop=True)

print(hc3_long["label"].value_counts())
print(hc3_long.groupby(["source", "label"]).size())

label
ai       5345
human    4775
Name: count, dtype: int64
source     label
finance    ai       4503
           human    3933
wiki_csai  ai        842
           human     842
dtype: int64


In [13]:
hc3_balanced = (hc3_long
    .groupby(["id", "label"], as_index=False)
    .first()
    .reset_index(drop=True))
print(hc3_balanced.groupby(["source", "label"]).size())

source     label
finance    ai       3933
           human    3933
wiki_csai  ai        842
           human     842
dtype: int64


In [14]:
hc3_balanced["question"] = (hc3_balanced["question"]
    .str.replace(r"\s*Please explain like I'm five\.?\s*$", "", regex=True))

In [15]:
hc3_balanced["text"] = hc3_balanced["text"].str.replace(r"URL_\d+", "", regex=True)

In [16]:
hc3_balanced["text"] = hc3_balanced["text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [17]:
hc3_balanced = hc3_balanced[hc3_balanced["text"].str.len() > 50].reset_index(drop=True)
print(hc3_balanced.shape)
print(hc3_balanced.groupby(["source", "label"]).size())

(9530, 5)
source     label
finance    ai       3929
           human    3917
wiki_csai  ai        842
           human     842
dtype: int64


In [18]:
# basic length statistics
hc3_balanced["char_count"] = hc3_balanced["text"].str.len()
hc3_balanced["word_count"] = hc3_balanced["text"].str.split().str.len()
hc3_balanced["sentence_count"] = hc3_balanced["text"].str.count(r"[.!?]+")

# group means
print(hc3_balanced.groupby(["source", "label"])[["char_count", "word_count", "sentence_count"]].mean())

                  char_count  word_count  sentence_count
source    label                                         
finance   ai     1225.146093  205.442352        9.024943
          human   996.369415  176.323207        9.891243
wiki_csai ai     1170.195962  185.337292        7.976247
          human  1298.710214  195.454869        9.226841


In [19]:
hc3_balanced.to_csv("hc3_clean.csv", index=False)